# NeceCity Model Training (Google Colab)

Trains the NeceCity YOLO detector on a free Colab GPU. This notebook runs the exact same `ml/training/train.py` script you can also run locally - nothing here is Colab-specific logic, it's just a convenient free GPU runtime.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> GPU (the free T4 tier is enough for `yolov8n`).

**You need:** your annotated dataset (see `ml/data/README.md`) uploaded somewhere Colab can reach it - either zipped in Google Drive, or pushed to a GitHub repo. This notebook shows the Google Drive path; adjust the "Get the dataset" cell if you're using GitHub instead.

## 1. Check GPU is available

In [ ]:
!nvidia-smi

## 2. Clone the NeceCity repository

In [ ]:
# Replace with your own fork/repo URL once you've pushed the project to GitHub.
!git clone https://github.com/<your-username>/nececity.git
%cd nececity/ml

## 3. Install ML dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Get your real dataset

Two common options - use whichever fits how you stored your annotated data:

In [ ]:
# Option A: dataset zipped in Google Drive
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/nececity_dataset.zip -d data/
# This should produce data/nececity_dataset/{images,labels}/{train,val}/ ...
# then edit training/dataset.yaml's `path:` to point at it, e.g.:
#   path: ../data/nececity_dataset

In [ ]:
# Option B: use the bundled synthetic demo dataset instead, just to prove
# the pipeline runs end-to-end on this Colab GPU (NOT a real model - see
# ml/data/README.md). Skip this cell if you ran Option A.
print('Using ml/data/demo_dataset as-is (synthetic placeholder data).')

## 5. (Optional) validate dataset structure before spending GPU time

In [ ]:
!python data/validate_dataset.py --dataset data/demo_dataset --classes data/classes.yaml

## 6. Train

Edit `configs/training_config.yaml` and `configs/model_config.yaml` first if you want to change epochs, batch size, or the initialization mode (pretrained vs. random).

In [ ]:
%cd training
!python train.py
%cd ..

## 7. Evaluate

In [ ]:
%cd evaluation
!python evaluate.py --weights ../training/runs/detect/nececity_yolo/weights/best.pt --dataset ../training/dataset.yaml --split val
%cd ..

## 8. Download the trained weights

Copy `best.pt` to `ml/weights/nececity_yolo.pt` in your local project (or upload it to your Drive), then set `MODEL_PATH` in `backend/.env` accordingly - see `ml/weights/README.md`.

In [ ]:
from google.colab import files
files.download('training/runs/detect/nececity_yolo/weights/best.pt')